# Phase 6: RAG & Vector Databases
## Day 30: RagChatbotProject

Date: 2026-04-24

### Learning objectives
- Build a full RAG chatbot over synthetic campaign data.
- Create campaign data inline.
- Convert rows into documents and chunks.
- Build local embeddings and retrieval.
- Answer questions with cited sources.
- Add simple chat memory.
- Evaluate retrieval and chatbot answers.

In [ ]:
import json
import re
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    TfidfVectorizer = None
    SKLEARN_AVAILABLE = False

def show(title, content):
    print("\n" + "=" * 84)
    print(title)
    print("=" * 84)
    print(textwrap.dedent(str(content)).strip())

np.set_printoptions(precision=4, suppress=True)

print("Setup complete.")
print("scikit-learn available:", SKLEARN_AVAILABLE)

In [ ]:
campaign_data = [
    {
        "campaign_id": "C001",
        "campaign_name": "Spring Coffee Push",
        "channel": "Instagram",
        "region": "Berlin",
        "spend_eur": 1200,
        "clicks": 3420,
        "conversions": 184,
        "sentiment": "positive",
        "notes": "Limited-time discount improved conversions. Strong CTR compared with previous coffee campaigns.",
        "recommendation": "Scale carefully and monitor cost per conversion."
    },
    {
        "campaign_id": "C002",
        "campaign_name": "Bank App Onboarding",
        "channel": "Email",
        "region": "Germany",
        "spend_eur": 800,
        "clicks": 980,
        "conversions": 42,
        "sentiment": "neutral",
        "notes": "Subject line was too generic. Call to action was not clear enough.",
        "recommendation": "Rewrite subject line and test a simpler onboarding message."
    },
    {
        "campaign_id": "C003",
        "campaign_name": "Yoga Studio Trial",
        "channel": "TikTok",
        "region": "Berlin",
        "spend_eur": 650,
        "clicks": 2100,
        "conversions": 165,
        "sentiment": "positive",
        "notes": "Short beginner-friendly videos performed best. Audience responded well to simple copy.",
        "recommendation": "Create more short videos and test new landing page copy."
    },
    {
        "campaign_id": "C004",
        "campaign_name": "Premium Card Upgrade",
        "channel": "LinkedIn",
        "region": "Germany",
        "spend_eur": 1500,
        "clicks": 740,
        "conversions": 31,
        "sentiment": "negative",
        "notes": "Cost per conversion was high. Audience may be too broad.",
        "recommendation": "Narrow audience targeting before increasing spend."
    },
    {
        "campaign_id": "C005",
        "campaign_name": "Summer Bagel Launch",
        "channel": "Instagram",
        "region": "Berlin",
        "spend_eur": 900,
        "clicks": 2800,
        "conversions": 220,
        "sentiment": "positive",
        "notes": "Food photos and short reels performed best. Strong conversion rate from local audience.",
        "recommendation": "Scale Instagram reels and reuse best-performing food visuals."
    },
    {
        "campaign_id": "C006",
        "campaign_name": "Invoice OCR Webinar",
        "channel": "LinkedIn",
        "region": "Germany",
        "spend_eur": 1100,
        "clicks": 1600,
        "conversions": 96,
        "sentiment": "positive",
        "notes": "Technical audience responded well to document intelligence examples and OCR pipeline content.",
        "recommendation": "Create a follow-up webinar about RAG over scanned documents."
    }
]

df = pd.DataFrame(campaign_data)
df["conversion_rate"] = df["conversions"] / df["clicks"]
df["cost_per_conversion"] = df["spend_eur"] / df["conversions"]

df

## 1. Project goal

This is the final capstone project.

You will build a small RAG chatbot that answers questions about campaign data using retrieval and source citations.

In [ ]:
project_steps = pd.DataFrame([
    {"step": 1, "stage": "Create data", "output": "Synthetic campaign table"},
    {"step": 2, "stage": "Document conversion", "output": "Text documents with metadata"},
    {"step": 3, "stage": "Chunking", "output": "Searchable chunks"},
    {"step": 4, "stage": "Embeddings", "output": "Vectors for chunks"},
    {"step": 5, "stage": "Retrieval", "output": "Top relevant chunks"},
    {"step": 6, "stage": "Prompting", "output": "Grounded answer prompt"},
    {"step": 7, "stage": "Chatbot", "output": "Answer with sources"},
    {"step": 8, "stage": "Evaluation", "output": "Retrieval and answer checks"},
])

project_steps

## 2. Convert rows into documents

RAG needs text documents.

Each campaign row becomes a document with readable text and useful metadata.

In [ ]:
def campaign_row_to_document(row):
    content = f'''
    Campaign ID: {row["campaign_id"]}
    Campaign Name: {row["campaign_name"]}
    Channel: {row["channel"]}
    Region: {row["region"]}
    Spend EUR: {row["spend_eur"]}
    Clicks: {row["clicks"]}
    Conversions: {row["conversions"]}
    Conversion Rate: {row["conversion_rate"]:.4f}
    Cost Per Conversion EUR: {row["cost_per_conversion"]:.2f}
    Sentiment: {row["sentiment"]}
    Notes: {row["notes"]}
    Recommendation: {row["recommendation"]}
    '''.strip()

    return {
        "page_content": textwrap.dedent(content),
        "metadata": {
            "campaign_id": row["campaign_id"],
            "campaign_name": row["campaign_name"],
            "channel": row["channel"],
            "region": row["region"],
            "source": "synthetic_campaign_table"
        }
    }

documents = [campaign_row_to_document(row) for _, row in df.iterrows()]

print("Number of documents:", len(documents))
pprint(documents[0]["metadata"])
print("\nDocument preview:")
print(documents[0]["page_content"])

## 3. Chunk the documents

Our campaign documents are short, so each document can be one chunk.

The function still supports splitting longer text.

In [ ]:
def recursive_split_text(text, chunk_size=600, separators=None):
    separators = separators or ["\n\n", "\n", ". ", " ", ""]
    text = text.strip()

    if len(text) <= chunk_size:
        return [text] if text else []

    separator = separators[0]
    remaining = separators[1:]

    if separator == "":
        return [text[i:i + chunk_size].strip() for i in range(0, len(text), chunk_size) if text[i:i + chunk_size].strip()]

    parts = text.split(separator)

    if len(parts) == 1:
        return recursive_split_text(text, chunk_size=chunk_size, separators=remaining)

    chunks = []
    current = ""

    for part in parts:
        if not part.strip():
            continue

        candidate = (current + separator + part).strip() if current else part.strip()

        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining))
            current = part.strip()

    if current:
        chunks.extend(recursive_split_text(current, chunk_size=chunk_size, separators=remaining))

    return chunks

def split_documents(documents, chunk_size=600):
    chunks = []

    for doc in documents:
        pieces = recursive_split_text(doc["page_content"], chunk_size=chunk_size)

        for index, piece in enumerate(pieces):
            metadata = {
                **doc["metadata"],
                "chunk_index": index,
                "chunk_id": f'{doc["metadata"]["campaign_id"]}-{index:03d}'
            }
            chunks.append({
                "page_content": piece,
                "metadata": metadata
            })

    return chunks

chunks = split_documents(documents, chunk_size=600)

print("Number of chunks:", len(chunks))
pprint(chunks[0]["metadata"])

In [ ]:
chunk_table = pd.DataFrame([
    {
        "chunk_id": chunk["metadata"]["chunk_id"],
        "campaign_id": chunk["metadata"]["campaign_id"],
        "campaign_name": chunk["metadata"]["campaign_name"],
        "channel": chunk["metadata"]["channel"],
        "chars": len(chunk["page_content"]),
        "preview": chunk["page_content"][:100] + "..."
    }
    for chunk in chunks
])

chunk_table

## 4. Build a simple vector store

The vector store embeds chunks and searches for similar chunks.

We use TF-IDF when available, with a tiny fallback if not.

In [ ]:
class CampaignVectorStore:
    def __init__(self):
        self.chunks = []
        self.matrix = None
        self.vectorizer = None
        self.vocabulary = None

    def _fallback_embedding(self, text):
        words = re.findall(r"[a-z0-9]+", text.lower())
        return np.array([words.count(term) for term in self.vocabulary], dtype=float)

    @staticmethod
    def cosine_similarity(a, b):
        denominator = np.linalg.norm(a) * np.linalg.norm(b)
        if denominator == 0:
            return 0.0
        return float(np.dot(a, b) / denominator)

    def add_chunks(self, chunks):
        self.chunks = chunks
        texts = [chunk["page_content"] for chunk in chunks]

        if SKLEARN_AVAILABLE:
            self.vectorizer = TfidfVectorizer(stop_words="english")
            self.matrix = self.vectorizer.fit_transform(texts).toarray()
        else:
            self.vocabulary = sorted(set(re.findall(r"[a-z0-9]+", " ".join(texts).lower())))
            self.matrix = np.vstack([self._fallback_embedding(text) for text in texts])

    def embed_query(self, query):
        if self.vectorizer is not None:
            return self.vectorizer.transform([query]).toarray()[0]
        return self._fallback_embedding(query)

    def similarity_search(self, query, k=3, metadata_filter=None):
        query_vector = self.embed_query(query)
        scores = [self.cosine_similarity(query_vector, row) for row in self.matrix]

        rows = []
        for chunk, score in zip(self.chunks, scores):
            if metadata_filter:
                keep = all(chunk["metadata"].get(key) == value for key, value in metadata_filter.items())
                if not keep:
                    continue

            rows.append({
                "page_content": chunk["page_content"],
                "metadata": chunk["metadata"],
                "score": score
            })

        return sorted(rows, key=lambda row: row["score"], reverse=True)[:k]

vectorstore = CampaignVectorStore()
vectorstore.add_chunks(chunks)

print("Stored chunks:", len(vectorstore.chunks))
print("Embedding matrix shape:", vectorstore.matrix.shape)

In [ ]:
results = vectorstore.similarity_search("Which campaign had a generic subject line?", k=3)

for item in results:
    print("\nScore:", round(item["score"], 4))
    print(item["metadata"]["chunk_id"], "|", item["metadata"]["campaign_name"])
    print(item["page_content"][:250])

## 5. Build the retriever

A retriever wraps search.

It controls top-k and optional metadata filters.

In [ ]:
class CampaignRetriever:
    def __init__(self, vectorstore, k=3, metadata_filter=None):
        self.vectorstore = vectorstore
        self.k = k
        self.metadata_filter = metadata_filter

    def get_relevant_documents(self, query):
        return self.vectorstore.similarity_search(
            query,
            k=self.k,
            metadata_filter=self.metadata_filter
        )

retriever = CampaignRetriever(vectorstore, k=3)

retrieved = retriever.get_relevant_documents("Which campaign should scale on Instagram?")

for item in retrieved:
    print(item["metadata"]["campaign_name"], "|", round(item["score"], 4))

In [ ]:
instagram_retriever = CampaignRetriever(
    vectorstore,
    k=3,
    metadata_filter={"channel": "Instagram"}
)

instagram_results = instagram_retriever.get_relevant_documents("Which campaign should scale?")

for item in instagram_results:
    print(item["metadata"]["campaign_name"], "|", item["metadata"]["channel"], "|", round(item["score"], 4))

## 6. Prompt template

The chatbot prompt should force grounded answers.

It should say to use only retrieved context and cite source chunk IDs.

In [ ]:
class PromptTemplate:
    def __init__(self, template):
        self.template = template

    def format(self, **kwargs):
        return self.template.format(**kwargs)

chatbot_prompt = PromptTemplate('''
You are a campaign analytics assistant.

Use only the context below to answer the question.
If the answer is not in the context, say: "I do not know from the provided context."

Context:
{context}

Question:
{question}

Answer rules:
- Keep the answer short.
- Mention campaign names when relevant.
- Include source chunk IDs.
'''.strip())

def format_context(retrieved_docs):
    blocks = []

    for item in retrieved_docs:
        metadata = item["metadata"]
        blocks.append(
            f'Source: {metadata["chunk_id"]} | Campaign: {metadata["campaign_name"]}\n'
            f'{item["page_content"]}'
        )

    return "\n\n---\n\n".join(blocks)

question = "Which campaign had the lowest conversion?"
context = format_context(retriever.get_relevant_documents(question))
prompt = chatbot_prompt.format(context=context, question=question)

show("Formatted chatbot prompt", prompt)

## 7. Mock LLM answer engine

A real chatbot would send the prompt to an LLM.

This notebook uses a deterministic mock engine so the project runs without API keys.

In [ ]:
def parse_campaigns_from_context(retrieved_docs):
    campaigns = []

    for item in retrieved_docs:
        text = item["page_content"]
        metadata = item["metadata"]

        def grab(pattern, cast=str):
            match = re.search(pattern, text)
            if not match:
                return None
            value = match.group(1).strip()
            return cast(value)

        campaigns.append({
            "campaign_id": metadata["campaign_id"],
            "chunk_id": metadata["chunk_id"],
            "campaign_name": metadata["campaign_name"],
            "channel": metadata["channel"],
            "region": metadata["region"],
            "spend_eur": grab(r"Spend EUR:\s*([0-9.]+)", float),
            "clicks": grab(r"Clicks:\s*([0-9]+)", int),
            "conversions": grab(r"Conversions:\s*([0-9]+)", int),
            "conversion_rate": grab(r"Conversion Rate:\s*([0-9.]+)", float),
            "cost_per_conversion": grab(r"Cost Per Conversion EUR:\s*([0-9.]+)", float),
            "sentiment": grab(r"Sentiment:\s*(.+)"),
            "notes": grab(r"Notes:\s*(.+)"),
            "recommendation": grab(r"Recommendation:\s*(.+)")
        })

    return campaigns

def mock_llm_answer(question, retrieved_docs):
    q = question.lower()
    campaigns = parse_campaigns_from_context(retrieved_docs)
    sources = [item["metadata"]["chunk_id"] for item in retrieved_docs]

    if not campaigns:
        return {"answer": "I do not know from the provided context.", "sources": sources}

    if "lowest conversion" in q or "lowest conversion rate" in q:
        campaign = min(campaigns, key=lambda x: x["conversion_rate"] if x["conversion_rate"] is not None else 999)
        return {
            "answer": f'{campaign["campaign_name"]} has the lowest conversion rate among retrieved campaigns: {campaign["conversion_rate"]:.2%}.',
            "sources": [campaign["chunk_id"]]
        }

    if "highest conversion" in q or "best conversion" in q:
        campaign = max(campaigns, key=lambda x: x["conversion_rate"] if x["conversion_rate"] is not None else -1)
        return {
            "answer": f'{campaign["campaign_name"]} has the highest conversion rate among retrieved campaigns: {campaign["conversion_rate"]:.2%}.',
            "sources": [campaign["chunk_id"]]
        }

    if "generic subject" in q:
        for campaign in campaigns:
            if campaign["notes"] and "generic" in campaign["notes"].lower():
                return {
                    "answer": f'{campaign["campaign_name"]} had a generic subject line. The recommendation is to rewrite it and test a simpler onboarding message.',
                    "sources": [campaign["chunk_id"]]
                }

    if "scale" in q:
        scale_candidates = [
            c for c in campaigns
            if c["recommendation"] and "scale" in c["recommendation"].lower()
        ]
        if scale_candidates:
            names = ", ".join(c["campaign_name"] for c in scale_candidates)
            return {
                "answer": f'The retrieved campaigns recommended for scaling are: {names}.',
                "sources": [c["chunk_id"] for c in scale_candidates]
            }

    if "cost per conversion" in q or "expensive" in q:
        campaign = max(campaigns, key=lambda x: x["cost_per_conversion"] if x["cost_per_conversion"] is not None else -1)
        return {
            "answer": f'{campaign["campaign_name"]} has the highest cost per conversion among retrieved campaigns: {campaign["cost_per_conversion"]:.2f} EUR.',
            "sources": [campaign["chunk_id"]]
        }

    if "ocr" in q or "document" in q:
        for campaign in campaigns:
            if "OCR" in campaign["campaign_name"] or "document" in (campaign["notes"] or ""):
                return {
                    "answer": f'{campaign["campaign_name"]} is related to OCR and document intelligence. Recommendation: {campaign["recommendation"]}',
                    "sources": [campaign["chunk_id"]]
                }

    return {
        "answer": "I do not know from the provided context.",
        "sources": sources
    }

answer = mock_llm_answer("Which campaign had a generic subject line?", retriever.get_relevant_documents("generic subject line"))
pprint(answer)

## 8. Build the RAG chatbot

The chatbot connects retriever, prompt template, and answer engine.

It returns the answer, sources, retrieved context, and prompt.

In [ ]:
class RagChatbot:
    def __init__(self, retriever, prompt_template, answer_function):
        self.retriever = retriever
        self.prompt_template = prompt_template
        self.answer_function = answer_function

    def ask(self, question):
        retrieved_docs = self.retriever.get_relevant_documents(question)
        context = format_context(retrieved_docs)
        prompt = self.prompt_template.format(context=context, question=question)
        answer_result = self.answer_function(question, retrieved_docs)

        return {
            "question": question,
            "answer": answer_result["answer"],
            "sources": answer_result["sources"],
            "retrieved_docs": retrieved_docs,
            "prompt": prompt
        }

chatbot = RagChatbot(
    retriever=retriever,
    prompt_template=chatbot_prompt,
    answer_function=mock_llm_answer
)

response = chatbot.ask("Which campaign had a generic subject line?")

print("Answer:")
print(response["answer"])
print("\nSources:", response["sources"])

In [ ]:
questions = [
    "Which campaign had the lowest conversion rate?",
    "Which campaign had the highest conversion rate?",
    "Which campaign had a generic subject line?",
    "Which campaigns should scale?",
    "Which campaign was most expensive by cost per conversion?",
    "Which campaign is related to OCR or document intelligence?"
]

for q in questions:
    result = chatbot.ask(q)
    print("\nQuestion:", q)
    print("Answer:", result["answer"])
    print("Sources:", result["sources"])

## 9. Source formatting

A useful chatbot should show sources clearly.

This helps users trust and verify the answer.

In [ ]:
def format_chatbot_response(result):
    source_lines = []

    for doc in result["retrieved_docs"]:
        metadata = doc["metadata"]
        if metadata["chunk_id"] in result["sources"]:
            source_lines.append(
                f'- {metadata["chunk_id"]}: {metadata["campaign_name"]} ({metadata["channel"]}, {metadata["region"]})'
            )

    if not source_lines:
        source_lines = [f'- {source}' for source in result["sources"]]

    return result["answer"] + "\n\nSources:\n" + "\n".join(source_lines)

formatted_response = format_chatbot_response(chatbot.ask("Which campaign had the highest conversion rate?"))
print(formatted_response)

In [ ]:
def inspect_retrieval(result):
    rows = []

    for item in result["retrieved_docs"]:
        metadata = item["metadata"]
        rows.append({
            "chunk_id": metadata["chunk_id"],
            "campaign_name": metadata["campaign_name"],
            "channel": metadata["channel"],
            "score": round(item["score"], 4),
            "used_as_source": metadata["chunk_id"] in result["sources"]
        })

    return pd.DataFrame(rows)

inspect_retrieval(chatbot.ask("Which campaign had a generic subject line?"))

## 10. Add simple memory

Memory stores previous turns.

It helps with conversation context, but retrieval still grounds the answer.

In [ ]:
class SimpleChatMemory:
    def __init__(self, max_turns=4):
        self.max_turns = max_turns
        self.turns = []

    def add(self, question, answer):
        self.turns.append({"question": question, "answer": answer})
        self.turns = self.turns[-self.max_turns:]

    def history_text(self):
        if not self.turns:
            return ""

        lines = []
        for turn in self.turns:
            lines.append(f'User: {turn["question"]}')
            lines.append(f'Assistant: {turn["answer"]}')
        return "\n".join(lines)

memory = SimpleChatMemory(max_turns=3)

first = chatbot.ask("Which campaign had the lowest conversion rate?")
memory.add(first["question"], first["answer"])

second = chatbot.ask("Which campaign had a generic subject line?")
memory.add(second["question"], second["answer"])

print(memory.history_text())

In [ ]:
class MemoryRagChatbot(RagChatbot):
    def __init__(self, retriever, prompt_template, answer_function, memory):
        super().__init__(retriever, prompt_template, answer_function)
        self.memory = memory

    def ask(self, question):
        result = super().ask(question)
        self.memory.add(question, result["answer"])
        result["memory"] = self.memory.history_text()
        return result

memory_chatbot = MemoryRagChatbot(
    retriever=retriever,
    prompt_template=chatbot_prompt,
    answer_function=mock_llm_answer,
    memory=SimpleChatMemory(max_turns=3)
)

memory_chatbot.ask("Which campaign had the highest conversion rate?")
memory_result = memory_chatbot.ask("Which campaign should scale?")

print(memory_result["answer"])
print("\nMemory:")
print(memory_result["memory"])

## 11. Retrieval evaluation

A RAG chatbot is only as good as its retrieval.

We check whether expected campaigns appear in top results.

In [ ]:
retrieval_tests = [
    {
        "query": "generic subject line",
        "expected_campaign_ids": {"C002"}
    },
    {
        "query": "TikTok beginner-friendly videos",
        "expected_campaign_ids": {"C003"}
    },
    {
        "query": "highest cost per conversion LinkedIn broad audience",
        "expected_campaign_ids": {"C004"}
    },
    {
        "query": "OCR document intelligence webinar",
        "expected_campaign_ids": {"C006"}
    },
    {
        "query": "Instagram food reels bagel",
        "expected_campaign_ids": {"C005"}
    }
]

def evaluate_retrieval(retriever, tests):
    rows = []

    for test in tests:
        docs = retriever.get_relevant_documents(test["query"])
        retrieved_ids = {doc["metadata"]["campaign_id"] for doc in docs}
        hit = len(retrieved_ids & test["expected_campaign_ids"]) > 0

        rows.append({
            "query": test["query"],
            "expected": sorted(test["expected_campaign_ids"]),
            "retrieved": sorted(retrieved_ids),
            "hit": hit
        })

    return pd.DataFrame(rows)

retrieval_eval = evaluate_retrieval(retriever, retrieval_tests)
retrieval_eval

In [ ]:
retrieval_hit_rate = retrieval_eval["hit"].mean()
print("Retrieval hit rate:", round(retrieval_hit_rate, 3))

## 12. Answer evaluation

For a simple project, start with rule-based answer checks.

In production, you can add human review or model-based evaluation.

In [ ]:
answer_tests = [
    {
        "question": "Which campaign had a generic subject line?",
        "must_contain": ["Bank App Onboarding"]
    },
    {
        "question": "Which campaign is related to OCR or document intelligence?",
        "must_contain": ["Invoice OCR Webinar"]
    },
    {
        "question": "Which campaign had the highest conversion rate?",
        "must_contain": ["Summer Bagel Launch"]
    }
]

def evaluate_answers(chatbot, tests):
    rows = []

    for test in tests:
        result = chatbot.ask(test["question"])
        answer = result["answer"]
        passed = all(term.lower() in answer.lower() for term in test["must_contain"])

        rows.append({
            "question": test["question"],
            "answer": answer,
            "must_contain": test["must_contain"],
            "passed": passed,
            "sources": result["sources"]
        })

    return pd.DataFrame(rows)

answer_eval = evaluate_answers(chatbot, answer_tests)
answer_eval

In [ ]:
answer_pass_rate = answer_eval["passed"].mean()
print("Answer pass rate:", round(answer_pass_rate, 3))

## 13. Build a final project report

A good final report shows data size, retrieval quality, answer quality, and next improvements.

In [ ]:
final_project_report = {
    "num_campaign_rows": len(df),
    "num_documents": len(documents),
    "num_chunks": len(chunks),
    "retrieval_hit_rate": round(float(retrieval_hit_rate), 3),
    "answer_pass_rate": round(float(answer_pass_rate), 3),
    "best_conversion_campaign": df.sort_values("conversion_rate", ascending=False).iloc[0]["campaign_name"],
    "lowest_conversion_campaign": df.sort_values("conversion_rate", ascending=True).iloc[0]["campaign_name"],
    "highest_cost_per_conversion_campaign": df.sort_values("cost_per_conversion", ascending=False).iloc[0]["campaign_name"],
    "recommended_next_steps": [
        "Replace TF-IDF with OpenAI embeddings or sentence-transformers.",
        "Store chunks in ChromaDB or FAISS.",
        "Add real LLM calls for natural language answers.",
        "Add more evaluation questions.",
        "Create a small UI with Streamlit or FastAPI."
    ]
}

pprint(final_project_report)

## 14. Optional production architecture

This is how the project can grow into a real app.

The notebook version is small, but the same architecture scales.

In [ ]:
production_architecture = pd.DataFrame([
    {"layer": "Data", "tool_options": "CSV, database, warehouse, Google Sheets"},
    {"layer": "Chunking", "tool_options": "LangChain splitter, custom splitter"},
    {"layer": "Embeddings", "tool_options": "OpenAI embeddings, sentence-transformers"},
    {"layer": "Vector store", "tool_options": "ChromaDB, FAISS, Pinecone, Weaviate"},
    {"layer": "LLM", "tool_options": "OpenAI, Ollama, Claude, local models"},
    {"layer": "API", "tool_options": "FastAPI, Flask"},
    {"layer": "UI", "tool_options": "Streamlit, React, iOS app"},
    {"layer": "Evaluation", "tool_options": "Human review, test queries, answer checks"},
])

production_architecture

In [ ]:
def production_readiness_check():
    checks = [
        "Do we have enough source data?",
        "Are chunks small enough but still meaningful?",
        "Are embeddings created with the same model for docs and queries?",
        "Does retrieval return expected chunks?",
        "Does the answer include sources?",
        "Does the prompt prevent unsupported claims?",
        "Do we log failed questions?",
        "Do we have privacy and security rules for user data?"
    ]
    return checks

for item in production_readiness_check():
    print("-", item)

## Tricky bits

RAG projects fail when retrieval, prompting, and evaluation are not separated.

Always inspect retrieved context before changing the LLM.

In [ ]:
tricky_bits = pd.DataFrame([
    {
        "problem": "Answer is wrong",
        "likely_cause": "Retrieved chunks were not relevant",
        "fix": "Inspect retrieval and improve embeddings or chunking"
    },
    {
        "problem": "Answer has no citation",
        "likely_cause": "Source metadata is missing",
        "fix": "Store chunk_id and source metadata"
    },
    {
        "problem": "Chatbot invents facts",
        "likely_cause": "Prompt does not restrict answer to context",
        "fix": "Tell the model to use only retrieved context"
    },
    {
        "problem": "Follow-up questions are weak",
        "likely_cause": "No memory or query rewriting",
        "fix": "Add memory and standalone question rewriting"
    },
    {
        "problem": "Good source is not found",
        "likely_cause": "Bad query, bad chunking, or weak embeddings",
        "fix": "Tune top_k, chunking, and embedding model"
    }
])

tricky_bits

In [ ]:
def diagnose_rag_problem(symptom):
    symptom = symptom.lower()

    if "wrong" in symptom or "bad answer" in symptom:
        return "Inspect retrieved chunks first."
    if "source" in symptom or "citation" in symptom:
        return "Check metadata and source formatting."
    if "invent" in symptom or "hallucinate" in symptom:
        return "Strengthen the prompt to use only context."
    if "follow" in symptom:
        return "Add memory or rewrite follow-up questions."
    if "not found" in symptom or "missing" in symptom:
        return "Tune retrieval, chunking, top_k, and embeddings."
    return "Check data, chunks, embeddings, retrieval, prompt, and evaluation."

for symptom in [
    "The answer is wrong",
    "There is no source citation",
    "The chatbot invents facts",
    "Follow-up questions fail",
    "The right campaign is not found"
]:
    print(symptom, "=>", diagnose_rag_problem(symptom))

## Trick questions

1. Should you blame the LLM first when a RAG answer is wrong?

<details>
<summary>Answer</summary>

No. First inspect the retrieved chunks. Bad retrieval often causes bad answers.

</details>

2. Why convert rows into text documents?

<details>
<summary>Answer</summary>

Embeddings work on text. A clear text representation helps retrieval.

</details>

3. Why keep chunk IDs?

<details>
<summary>Answer</summary>

Chunk IDs make answers traceable and easier to debug.

</details>

4. Is TF-IDF enough for production RAG?

<details>
<summary>Answer</summary>

Usually no. It is useful for learning and simple baselines, but semantic embeddings are better for many production cases.

</details>

5. What should the final chatbot return besides the answer?

<details>
<summary>Answer</summary>

It should return sources, retrieved chunks, and useful metadata for debugging.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Convert the first campaign row into a document.

doc = ___

assert "page_content" in doc
assert "metadata" in doc
assert doc["metadata"]["campaign_id"] == "C001"
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Split all documents into chunks.

exercise_chunks = ___

assert isinstance(exercise_chunks, list)
assert len(exercise_chunks) >= len(documents)
assert "chunk_id" in exercise_chunks[0]["metadata"]
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Create a vector store and add chunks.

store = CampaignVectorStore()
___

assert store.matrix is not None
assert len(store.chunks) == len(exercise_chunks)
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Create a retriever with k=2.

exercise_retriever = ___

assert exercise_retriever.k == 2
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Retrieve chunks for a generic subject line query.

retrieved = ___

assert isinstance(retrieved, list)
assert len(retrieved) == 2
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Format retrieved context.

context_text = ___

assert isinstance(context_text, str)
assert "Source:" in context_text
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Build a RAG chatbot.

exercise_chatbot = ___

assert hasattr(exercise_chatbot, "ask")
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Ask the chatbot a question.

answer_result = ___

assert "answer" in answer_result
assert "sources" in answer_result
print("Exercise 8 passed.")

In [ ]:
# Exercise 9
# Evaluate retrieval.

eval_df = ___

assert isinstance(eval_df, pd.DataFrame)
assert "hit" in eval_df.columns
print("Exercise 9 passed.")

In [ ]:
# Exercise 10
# Format the chatbot response with sources.

formatted = ___

assert "Sources:" in formatted
print("Exercise 10 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
doc = campaign_row_to_document(df.iloc[0])
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
exercise_chunks = split_documents(documents, chunk_size=600)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
store.add_chunks(exercise_chunks)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
exercise_retriever = CampaignRetriever(store, k=2)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
retrieved = exercise_retriever.get_relevant_documents("generic subject line")
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
context_text = format_context(retrieved)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
exercise_chatbot = RagChatbot(
    retriever=exercise_retriever,
    prompt_template=chatbot_prompt,
    answer_function=mock_llm_answer
)
```

</details>

<details>
<summary>Exercise 8 solution</summary>

```python
answer_result = exercise_chatbot.ask("Which campaign had a generic subject line?")
```

</details>

<details>
<summary>Exercise 9 solution</summary>

```python
eval_df = evaluate_retrieval(exercise_retriever, retrieval_tests)
```

</details>

<details>
<summary>Exercise 10 solution</summary>

```python
formatted = format_chatbot_response(answer_result)
```

</details>

## Cumulative review exercises

These mix topics from Days 20 to 29. Fill in `___` and run each cell.

In [ ]:
# Review 1: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 80}
conversion_rate = ___

assert abs(conversion_rate - 0.08) < 1e-9
print("Review 1 passed.")

In [ ]:
# Review 2: Tesseract basics
# Choose the English language code.

english_lang_code = ___

assert english_lang_code == "eng"
print("Review 2 passed.")

In [ ]:
# Review 3: EasyOCR
# Create a language list for English and German.

easyocr_languages = ___

assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 3 passed.")

In [ ]:
# Review 4: OpenCV preprocessing
# Choose the thresholding method useful for uneven lighting.

threshold_method = ___

assert threshold_method.lower() == "adaptive"
print("Review 4 passed.")

In [ ]:
# Review 5: OCR plus LLM pipeline
# Choose the structured output format.

structured_format = ___

assert structured_format.upper() == "JSON"
print("Review 5 passed.")

In [ ]:
# Review 6: Document intelligence
# Create a quality rule for invoices.

quality_rule = ___

assert "total" in quality_rule.lower() or "date" in quality_rule.lower() or "id" in quality_rule.lower()
print("Review 6 passed.")

In [ ]:
# Review 7: Embeddings
# Calculate cosine similarity.

a = np.array([1, 0, 0])
b = np.array([1, 1, 0])
score = ___

assert 0.70 < score < 0.72
print("Review 7 passed.")

In [ ]:
# Review 8: Chunking
# Split text into chunks.

sample_text = "A RAG system retrieves relevant chunks before generating an answer."
chunked = ___

assert isinstance(chunked, list)
assert len(chunked) > 0
print("Review 8 passed.")

In [ ]:
# Review 9: Chroma and FAISS
# Choose what a vector database stores.

vector_db_stores = ___

assert "embedding" in vector_db_stores.lower() or "vector" in vector_db_stores.lower()
print("Review 9 passed.")

In [ ]:
# Review 10: LangChain RetrievalQA
# Name the component that returns relevant documents.

component = ___

assert component.lower() == "retriever"
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
conversion_rate = record["conversions"] / record["clicks"]

# Review 2
english_lang_code = "eng"

# Review 3
easyocr_languages = ["en", "de"]

# Review 4
threshold_method = "adaptive"

# Review 5
structured_format = "JSON"

# Review 6
quality_rule = "Invoice total must be present and non-negative."

# Review 7
score = CampaignVectorStore.cosine_similarity(a, b)

# Review 8
chunked = recursive_split_text(sample_text, chunk_size=40)

# Review 9
vector_db_stores = "embeddings and metadata"

# Review 10
component = "retriever"
```

</details>

In [ ]:
cheat_sheet = '''
DAY 30 CHEAT SHEET: RAG CHATBOT PROJECT

Full RAG chatbot steps:
1. Create or load source data.
2. Convert rows or files into text documents.
3. Add metadata to each document.
4. Split documents into chunks.
5. Create embeddings for chunks.
6. Store chunks in a vector store.
7. Retrieve top-k chunks for each question.
8. Format a prompt with context and question.
9. Generate an answer using only retrieved context.
10. Return answer, sources, and debug info.

Important metadata:
- campaign_id
- campaign_name
- channel
- region
- chunk_id
- source

Evaluation:
- Retrieval hit rate checks if the right chunks are found.
- Answer pass rate checks if the answer contains expected facts.
- Always inspect retrieved chunks when answers are wrong.

Production next steps:
- Replace TF-IDF with semantic embeddings.
- Use ChromaDB or FAISS.
- Add a real LLM.
- Add query rewriting for follow-up questions.
- Add logging and human review.
'''

print(cheat_sheet)

## Curriculum complete

You finished Day 30 — RagChatbotProject.

Next step: turn this notebook into a small portfolio project with a Streamlit or FastAPI interface.